In [1]:
%pip install -q librosa numpy

import os
import json
import numpy as np 
import librosa # For audio processing
from pathlib import Path
from typing import Tuple, List, Dict
import pickle # For saving and loading data

Note: you may need to restart the kernel to use updated packages.


In [ ]:
class AudoPreprocessor:
    # 30s audio clips at 16kHz
    def __init__(self, data_dir, output_dir, epoch_duration = 30, sample_rate = 16000):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.epoch_duration = epoch_duration
        self.sample_rate = sample_rate
        
        # AST Input Shape
        self.target_length = sample_rate * epoch_duration
        
        # Make output directory if it doesn't exist
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\nInitialized AudoPreprocessor")
        print(f"    Data_dir: {data_dir}")
        print(f"    Output_dir: {output_dir}")
        print(f"    Epoch Duration: {epoch_duration}s")
        print(f"    Sample Rate: {sample_rate}Hz")
        print(f"    Samples per epoch: {self.target_length}")
        
    def load_annotations(self, folder_id) -> Dict:
        # load annotations from json like 01_annotation.json
        annotation_path = self.data_dir / folder_id / f"{folder_id}_annotation.json"
        
        if (not annotation_path.exists()):
            raise FileNotFoundError(f"Annotation file not found: {annotation_path}")
        
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)
            
        # Debug print statements
        print(f"\nLoaded annotations from {annotation_path}")
        print(f"    Record Start: {annotations['record_start']}s")
        print(f"    Awake Intervals: {len(annotations['awake_intervals'])}")
        print(f"    Events: {len(annotations['events'])}")
        
        return annotations
    
    def load_audio(self, folder_id) -> Tuple[np.ndarray, int]:
        # load audio file like 01_phone.wav
        audio_path = self.data_dir / folder_id / f"{folder_id}_phone.wav"
        
        if (not audio_path.exists()):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        
        # Librosa audio loading
        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        
        # Debug print statements
        print(f"\nLoaded audio from {audio_path}")
        print(f"    Audio Shape: {audio.shape}")
        print(f"    Sample Rate: {sr}Hz")
        print(f"    Duration: {len(audio)/sr:.2f}s")
        
        return audio, sr
    
    # Check if time point is within any awake interval
    def is_awake(self, time_point: float, awake_intervals: List[Tuple[float]]) -> bool:
        for start, end in awake_intervals:
            if start <= time_point <= end:
                return True
        return False
        
    # Extract epoch labels based on annotations
    def extract_epoch_labels(self, epoch_start: float, epoch_end: float, events: List[Dict], awake_intervals: List[List[float]]) -> int:
        # Check if epoch during awake interval
        if (self.is_awake(epoch_start, awake_intervals) or self.is_awake(epoch_end, awake_intervals)):
            return -1  # Awake
        
        label = 0 # Default to no event (1 = osa [obstructive sleep apnea], 2 = hyp [hypnopnea])
        
        # Gonna prioritize in order hypo > osa > none
        for event in events:
            event_start = event['evnet_start']  # Note: typo in original data
            event_end = event_start + event['event_duration']
            event_type = event['event_type']
            
            # Check if event overlaps with epoch
            if not (event_end < epoch_start or event_start > epoch_end):
                if event_type == 'hypo':
                    label = max(label, 2)
                elif event_type == 'osa':
                    label = max(label, 1)
        
        return label
    
    # Create epochs from audio data and label them
    def create_epochs(self, folder_id: str) -> Tuple[List[np.ndarray], List[int]]:
        # Load data
        annotations = self.load_annotations(folder_id)
        audio, sr = self.load_audio(folder_id)
        
        # Extract Annotations
        record_start = annotations['record_start']
        awake_intervals = annotations['awake_intervals']
        events = annotations['events']
        
        # Number of epochs
        audio_duration = len(audio) / sr
        num_epochs = int(np.floor(audio_duration / self.epoch_duration))
        
        print (f"\nCreating {num_epochs} epochs of {self.epoch_duration}s each from audio of duration {audio_duration:.2f}s")
        
        epochs = []
        labels = []
        label_counts = { -1: 0, 0: 0, 1: 0, 2: 0 } # Awake, No Event, OSA, Hypo
        
        for i in range(num_epochs):
            epoch_start_sample = i * self.target_length
            epoch_end_sample = (i + 1) * self.target_length
            
            # Handle last epoch case if it too short
            if epoch_end_sample > len(audio):
                break
            
            epoch_audio = audio[epoch_start_sample:epoch_end_sample]
            
            # Actual start and end time per recording start
            epoch_start_time = record_start + (i * self.epoch_duration)
            epoch_end_time = epoch_start_time + self.epoch_duration
            
            label = self.extract_epoch_labels(epoch_start_time, epoch_end_time, events, awake_intervals)
            
            # Don't care if awake
            if label == -1:
                label_counts[-1] += 1
                continue
            
            epochs.append(epoch_audio)
            labels.append(label)
            label_counts[label] += 1
            
        print(f"\nEpoch Statistics for folder {folder_id}:")
        print(f"    Total Epochs: {num_epochs}")
        print(f"    Processed Epochs Saved: {len(epochs)}")
        print(f"    Awake Epochs Skipped: {label_counts[-1]}")
        print(f"    No Event Epochs: {label_counts[0]}")
        print(f"    OSA Event Epochs: {label_counts[1]}")
        print(f"    Hypopnea Event Epochs: {label_counts[2]}")
        
        return epochs, labels
    
    def process_folders(self, folder_ids: List[str] = None):
        
        # If no folder IDs provided, process all folders in data_dir
        if (not folder_ids):
            print ("\nNo folder IDs provided. Defaulting to folders 01-50.")
            folder_ids = [f"{i:02d}" for i in range(1, 51)] # Folders named 01 to 50
        
        all_epochs = []
        all_labels = []
        all_folder_ids = []
        
        print(f"\n{'='*40}")
        print(f"Processing folders {len(folder_ids)} folders")
        print(f"{'='*40}")
        
        for folder_id in folder_ids:
            print(f"\nProcessing folder {folder_id}...")
            
            epochs, labels = self.create_epochs(folder_id)
            
            all_epochs.extend(epochs)
            all_labels.extend(labels)
            all_folder_ids.extend([folder_id] * len(epochs))
            
            print(f"Completed processing folder {folder_id}. Total epochs so far: {len(all_epochs)}")
            
        # Save all processed data
        print(f"\n{'='*40}")
        print(f"Saving preprocessed data...")
        print(f"{'='*40}")
        
        data = {
            'epochs': np.array(all_epochs),
            'labels': np.array(all_labels),
            'folder_ids': all_folder_ids,
            'sample_rate': self.sample_rate,
            'epoch_duration': self.epoch_duration
        }
        
        output_path = self.output_dir / "preprocessed_data.pkl"
        with open(output_path, 'wb') as f:
            pickle.dump(data, f)
            
        print(f"\nPreprocessed data saved to {output_path}")
        print(f"    Total Epochs Saved: {len(all_epochs)}")
        print(f"    No-event Epochs: {all_labels.count(0)}")
        print(f"    OSA Event Epochs: {all_labels.count(1)}")
        print(f"    Hypopnea Event Epochs: {all_labels.count(2)}")
        print(f"    Data Shape: {data['epochs'].shape}")
        
        return data

if __name__ == "__main__":
    parent_dir = os.path.dirname(os.getcwd())
    
    # Define paths relative to the current directory
    DATA_DIR = parent_dir + "/Data"
    OUTPUT_DIR = parent_dir + "/Preprocessed"
    
    preprocessor = AudoPreprocessor(
        data_dir=DATA_DIR,
        output_dir=OUTPUT_DIR,
        epoch_duration=30,
        sample_rate=16000
    )
    
    test_folder_ids = [f"{i:02d}" for i in range(1, 51)]
    
    data = preprocessor.process_folders(folder_ids=test_folder_ids)


Initialized AudoPreprocessor
    Data_dir: c:\Users\jacst\Downloads\School\Fall25\CSE575/Data
    Output_dir: c:\Users\jacst\Downloads\School\Fall25\CSE575/Preprocessed
    Epoch Duration: 30s
    Sample Rate: 16000Hz
    Samples per epoch: 480000

Processing folders 2 folders

Processing folder 01...

Loaded annotations from c:\Users\jacst\Downloads\School\Fall25\CSE575\Data\01\01_annotation.json
    Record Start: 75454.0s
    Awake Intervals: 23
    Events: 86

Loaded audio from c:\Users\jacst\Downloads\School\Fall25\CSE575\Data\01\01_phone.wav
    Audio Shape: (356126720,)
    Sample Rate: 16000Hz
    Duration: 22257.92s

Creating 741 epochs of 30s each from audio of duration 22257.92s

Epoch Statistics for folder 01:
    Total Epochs: 741
    Processed Epochs Saved: 676
    Awake Epochs Skipped: 65
    No Event Epochs: 566
    OSA Event Epochs: 1
    Hypopnea Event Epochs: 109
Completed processing folder 01. Total epochs so far: 676

Processing folder 02...

Loaded annotations fro

In [1]:
# Using CUDA 13.0
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
# pip install transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig, ASTFeatureExtractor

c:\Users\jacst\anaconda3\envs\AST\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MultiEpochSleepApneaDetector(nn.Module):
    # Apnea Detector using Audio Spectrogram Transformer (AST)
    
    # Studies 14->10 Architecture
    # 14 contextual epochs to predict 10 output epochs
    def __init__(self, context_epochs=14, output_epochs=10, num_classes=3, dropout=0.3):
        super().__init__()
        
        self.context_epochs = context_epochs
        self.output_epochs = output_epochs
        self.num_classes = num_classes
        
        print(f"\nInitializing MultiEpochSleepApneaDetector: ")
        print(f"    Context Epochs: {context_epochs}")
        print(f"    Output Epochs: {output_epochs}")
        print(f"    Number of Classes: {num_classes}")
        print(f"    Dropout: {dropout}")
        
        # Load pre-trained AST model
        self.ast_config = ASTConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
                
        ast_feature_dim = self.ast_config.hidden_size #768 Dim Features
        
        # Multi-epoch temporal modeling with LSTM
        self.temporal_lstm = nn.LSTM(
            input_size=ast_feature_dim,
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            dropout=dropout if context_epochs > 1 else 0,
            bidirectional=True
        )
        
        # Attention layer to focus on relevant time steps
        self.attention = nn.MultiheadAttention(
            embed_dim=1024, # 512 * 2 for bidirectional
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Classification layer
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
        print(f"\nModel initialized successfully.")
        print(f"AST Feature Dimension: {ast_feature_dim}")
        
    def extract_ast_features(self, waveforms):
        batch_size, n_epochs, n_samples = waveforms.shape
        waveforms_flat = waveforms.reshape(batch_size * n_epochs, n_samples)
        
        # Convert to numpy and feed into extractor
        feature_extractor = get_ast_feature_extractor()
        inputs = feature_extractor(
            [w.cpu().numpy() for w in waveforms_flat], 
            sampling_rate=16000, 
            return_tensors="pt"
        )
        
        # Move to model device (CPU -> GPU if available)
        inputs = {k: v.to(next(self.parameters()).device) for k, v in inputs.items()}
        
        # Run AST on Inputs
        outputs = self.ast_model(**inputs)
        features = outputs.last_hidden_state[:, 0, :] # CLS token
            
        features = features.reshape(batch_size, n_epochs, -1)
        return features
    
    def forward(self, waveforms):
        # waveforms shape: (batch_size, context_epochs, samples_per_epoch)
        batch_size = waveforms.shape[0]
        
        # AST Feature Extraction
        features = self.extract_ast_features(waveforms) # (batch_size, context_epochs, feature_dim [768])

        # Temporal Info
        lstm_out, _ = self.temporal_lstm(features) # (batch_size, context_epochs, 1024)
        
        # Apply Attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out) # (batch_size, context_epochs, 1024)
        
        # Select middle output_epochs for prediction
        start_idx = (self.context_epochs - self.output_epochs) // 2
        end_idx = start_idx + self.output_epochs
        output_features = attn_out[:, start_idx:end_idx, :] # (batch_size, output_epochs, 1024)
        
        # Classify output
        logits = self.classifier(output_features) # (batch_size, output_epochs, num_classes)
        
        return logits
    
class WeightedCrossEntropyLoss(nn.Module):
    # Weighted Cross Entropy Loss for class imbalance
    # Study --> 1.0 for no, 1.3 for apnea, 2.1 for hypopnea
    
    def __init__(self, weights=None):
        super().__init__()
        
        if weights is None:
            weights = torch.tensor([1.0, 1.3, 2.1]) # Default weights
            
        self.weights = weights
        print(f"\nInitialized WeightedCrossEntropyLoss with weights: {self.weights}")
        
    def forward(self, logits, targets):
        # logits shape: (batch, epochs, num_classes)
        # targets shape: (batch, epochs)
        
        weights = self.weights.to(logits.device)
        
        #Reshape
        batch_size, n_epochs, n_classes = logits.shape
        logits_flat = logits.reshape(-1, n_classes)
        targets_flat = targets.reshape(-1)
        
        #Calc
        loss = F.cross_entropy(logits_flat, targets_flat, weight=weights)
        
        return loss
    
def get_ast_feature_extractor():
    feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
    return feature_extractor


#TESTING FUNCTION IGNORE (MADE IT WITH AI TO TEST THE MODEL ARCHITECTURE)
'''
if __name__ == "__main__":
    # Model Parms
    CONTEXT_EPOCHS = 14
    OUTPUT_EPOCHS = 10
    NUM_CLASSES = 3
    SAMPLE_RATE = 16000
    EPOCH_DURATION = 30 # seconds
    
    # CUDA
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device} and Cuda Version: {torch.version.cuda}")
        
    # Dummy Input
    batch_size = 2
    samples_per_epoch = SAMPLE_RATE * EPOCH_DURATION
    
    print(f"\nCreating dummy input:")
    print(f"  Batch size: {batch_size}")
    print(f"  Context epochs: {CONTEXT_EPOCHS}")
    print(f"  Samples per epoch: {samples_per_epoch}")
    
    x = torch.randn(batch_size, CONTEXT_EPOCHS, samples_per_epoch)
    y = torch.randint(0, NUM_CLASSES, (batch_size, OUTPUT_EPOCHS))
    
    print(f"  Input shape: {x.shape}")
    print(f"  Target shape: {y.shape}")
    
    # Initialize model
    print("\n" + "="*60)
    model = MultiEpochSleepApneaDetector(
        context_epochs=CONTEXT_EPOCHS,
        output_epochs=OUTPUT_EPOCHS,
        num_classes=NUM_CLASSES,
    )
    
    # Test forward pass
    print("\n" + "="*60)
    print("Testing forward pass...")
    print("="*60)
    
    model.eval()
    with torch.no_grad():
        logits = model(x)
    
    print(f"\nOutput shape: {logits.shape}")
    print(f"Expected shape: ({batch_size}, {OUTPUT_EPOCHS}, {NUM_CLASSES})")
    
    # Test loss functions
    print("\n" + "="*60)
    print("Testing loss functions...")
    print("="*60)
    
    criterion = WeightedCrossEntropyLoss(
        weights=torch.tensor([1.0, 1.3, 2.1])
    )
    
    loss = criterion(logits, y)
    print(f"\nLoss value: {loss.item():.4f}")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("\n" + "="*60)
    print("Model Statistics:")
    print("="*60)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB")
    
    print("\n" + "="*60)
    print("Model architecture test complete!")
    print("="*60)
'''

'\nif __name__ == "__main__":\n    # Model Parms\n    CONTEXT_EPOCHS = 14\n    OUTPUT_EPOCHS = 10\n    NUM_CLASSES = 3\n    SAMPLE_RATE = 16000\n    EPOCH_DURATION = 30 # seconds\n\n    # CUDA\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"\nUsing device: {device} and Cuda Version: {torch.version.cuda}")\n\n    # Dummy Input\n    batch_size = 2\n    samples_per_epoch = SAMPLE_RATE * EPOCH_DURATION\n\n    print(f"\nCreating dummy input:")\n    print(f"  Batch size: {batch_size}")\n    print(f"  Context epochs: {CONTEXT_EPOCHS}")\n    print(f"  Samples per epoch: {samples_per_epoch}")\n\n    x = torch.randn(batch_size, CONTEXT_EPOCHS, samples_per_epoch)\n    y = torch.randint(0, NUM_CLASSES, (batch_size, OUTPUT_EPOCHS))\n\n    print(f"  Input shape: {x.shape}")\n    print(f"  Target shape: {y.shape}")\n\n    # Initialize model\n    print("\n" + "="*60)\n    model = MultiEpochSleepApneaDetector(\n        context_epochs=CONTEXT_EPOCHS,\n        outp

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime
from sklearn.metrics import f1_score
import os

In [ ]:
class SleepApneaDataset(Dataset):
    def __init__(self, epochs, labels, folder_ids, context_epochs=14, output_epochs=10):
        self.epochs = epochs
        self.labels = labels
        self.folder_ids = folder_ids
        self.context_epochs = context_epochs
        self.output_epochs = output_epochs
        
        self.valid_indices = self._create_sequences()
        
        print(f"\nInitialized SleepApneaDataset:")
        print(f"    Total epochs: {len(self.epochs)}")
        print(f"    Valid sequences: {len(self.valid_indices)}")
        print(f"    Context epochs: {self.context_epochs}")
        print(f"    Output epochs: {self.output_epochs}")
        
    def _create_sequences(self):
        valid_indices = []
        total_epochs = len(self.epochs)
        
        for i in range(total_epochs - self.context_epochs + 1):
            folder_ids_in_seq = self.folder_ids[i:i + self.context_epochs]
            if (len(set(folder_ids_in_seq)) == 1):
                valid_indices.append(i)
        return valid_indices
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        start_idx = self.valid_indices[idx]
        
        context = self.epochs[start_idx:start_idx + self.context_epochs]
        
        label_start = start_idx + (self.context_epochs - self.output_epochs) // 2
        label_end = label_start + self.output_epochs
        labels = self.labels[label_start:label_end]
        
        context_tensor = torch.FloatTensor(context)
        labels_tensor = torch.LongTensor(labels)
        
        return context_tensor, labels_tensor
    
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, device, output_dir, patience=10):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.output_dir = Path(output_dir)
        self.patience = patience
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # History
        self.history = {
            'train_loss': [],
            'train_accuracy': [],
            'val_loss': [],
            'val_accuracy': [],
            'val_f1': []
        }
        
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        self.epochs_no_improve = 0
        
        print(f"\nInitialized Trainer:")
        print(f"    Device: {self.device}")
        print(f"    Output Directory: {self.output_dir}")
        print(f"    Training Batches: {len(self.train_loader)}")
        print(f"    Val Batches: {len(self.val_loader)}")
        print(f"    Patience: {self.patience} epochs")
        
    def train_epoch(self, epoch):
        self.model.train()
        
        total_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch} [Training]", leave=False)
        
        for batch_idx, (data, targets) in enumerate(pbar):
            data, targets = data.to(self.device), targets.to(self.device)
            
            # Forward
            self.optimizer.zero_grad()
            logits = self.model(data)
            
            # Loss
            loss = self.criterion(logits, targets)
            
            # Backward
            loss.backward()
            
            # Clipping Gradients
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            # Update
            self.optimizer.step()
            
            # Statistics
            total_loss += loss.item()
            predictions = logits.argmax(dim=-1)
            correct += (predictions == targets).sum().item()
            total += targets.numel()
            
            pbar.set_postfix({
                'Loss': f"{total_loss:.4f}",
                'Accuracy': f"{100.0 * correct / total:.2f}%"
            })
            
        avg_loss = total_loss / len(self.train_loader)
        accuracy = 100.0 * correct / total
        
        return avg_loss, accuracy
    
    def validate_epoch(self, epoch):
        self.model.eval()
        
        total_loss = 0.0
        correct = 0
        total = 0
        
        # F1 Score Calculation Stuff
        all_predictions = []
        all_targets = []
        
        pbar = tqdm(self.val_loader, desc=f"Epoch {epoch} [Validation]", leave=False)
        
        with torch.no_grad():
            for data, targets in pbar:
                data, targets = data.to(self.device), targets.to(self.device)
                
                # Forward
                logits = self.model(data)
                
                # Loss
                loss = self.criterion(logits, targets)
                
                # Statistics
                total_loss += loss.item()
                predictions = logits.argmax(dim=-1)
                correct += (predictions == targets).sum().item()
                total += targets.numel()
                
                all_predictions.extend(predictions.cpu().numpy().flatten())
                all_targets.extend(targets.cpu().numpy().flatten())
                
                pbar.set_postfix({
                    'Loss': f"{total_loss:.4f}",
                    'Accuracy': f"{100.0 * correct / total:.2f}%"
                })
        
        avg_loss = total_loss / len(self.val_loader)
        accuracy = 100.0 * correct / total
        
        # Calculate F1 Score
        f1 = f1_score(all_targets, all_predictions, average='macro', zero_division=0)
        
        return avg_loss, accuracy, f1, all_predictions, all_targets
    
    def train(self, num_epochs):
        print("\n" + "="*40)
        print(f"Starting training for {num_epochs} epochs...")
        print("="*40)
        
        for epoch in range(1, num_epochs + 1):
            print (f"\nEpoch {epoch}/{num_epochs}")
            print("-"*40)
            
            train_loss, train_acc = self.train_epoch(epoch)
            
            val_loss, val_accuracy, val_f1, _, _ = self.validate_epoch(epoch)
            
            # Log history
            self.history['train_loss'].append(train_loss)
            self.history['train_accuracy'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_accuracy'].append(val_accuracy)
            self.history['val_f1'].append(val_f1)
            
            print(f"\nEpoch {epoch} Summary:")
            print(f"    Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%")
            print(f"    Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%, Val F1: {val_f1:.4f}")
            
            # Check for improvement
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.best_epoch = epoch
                self.epochs_no_improve = 0
                
                checkpoint_path = self.output_dir / "best_model.pth"
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_loss': val_loss,
                    'val_accuracy': val_accuracy,
                    'val_f1': val_f1
                }, checkpoint_path)
                
                # Save best model
                print(f"    New best model saved to best_model.pth (Val Loss: {val_loss:.4f})")
            else:
                self.epochs_no_improve += 1
                print(f"    No improvement for {self.epochs_no_improve} epochs.")
                
            # Early stopping
            if self.epochs_no_improve >= self.patience:
                print(f"\nEarly stopping triggered after {self.patience} epochs with no improvement.")
                print(f"Best model was from epoch {self.best_epoch} with Val Loss: {self.best_val_loss:.4f}")
                break
            
            # Checkpoints
            if epoch % 5 == 0:
                checkpoint_path = self.output_dir / f'checkpoint_epoch_{epoch}.pth'
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'history': self.history
                }, checkpoint_path)
                print(f"    Checkpoint saved to {checkpoint_path}")
        
        # Save training history
        history_path = self.output_dir / "training_history.json"
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f"\nTraining history saved to {history_path}")
        
        print(f"\n {'='*40}")
        print(f"Training Complete!")
        print(f"   Total epochs trained: {epoch}")
        print(f"   Best epoch: {self.best_epoch} with Val Loss: {self.best_val_loss:.4f}")
        
if __name__ == "__main__":
    parent_dir = os.path.dirname(os.getcwd())
    
    # Define paths relative to the current directory
    DATA_PATH = parent_dir + "/Preprocessed/preprocessed_data.pkl"
    OUTPUT_DIR = parent_dir + "/Model_Output"
    
    # Parms
    CONTEXT_EPOCHS = 14
    OUTPUT_EPOCHS = 10
    NUM_CLASSES = 3
    BATCH_SIZE = 8
    NUM_EPOCHS = 1 # Increase for real training to 50
    LEARNING_RATE = 1e-4
    PATIENCE = 10
    VAL_SPLIT = 0.2
    
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device} and Cuda Version: {torch.version.cuda}")
    
    # Load preprocessed data
    print(f"\nLoading preprocessed data from {DATA_PATH}...")
    with open(DATA_PATH, 'rb') as f:
        data = pickle.load(f)
    print(f"    Total epochs: {data['epochs'].shape}")
    print(f"    Total labels: {data['labels'].shape}")
    
    # Create Dataset
    dataset = SleepApneaDataset(
        epochs=data['epochs'],
        labels=data['labels'],
        folder_ids=data['folder_ids'],
        context_epochs=CONTEXT_EPOCHS,
        output_epochs=OUTPUT_EPOCHS
    )
    
    # Train/Val Split
    val_size = int(VAL_SPLIT * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(
        dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"\nDataset split:")
    print(f"    Training samples: {len(train_dataset)}")
    print(f"    Validation samples: {len(val_dataset)}")
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=0,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    # Initialize model
    print("\n" + "="*40)
    print("Initializing model...")
    print("="*40)
    
    model = MultiEpochSleepApneaDetector(
        context_epochs=CONTEXT_EPOCHS,
        output_epochs=OUTPUT_EPOCHS,
        num_classes=NUM_CLASSES,
        dropout=0.3
    ).to(device)
    
    # Loss and Optimizer
    criterion = WeightedCrossEntropyLoss(
        weights=torch.tensor([1.0, 1.3, 2.1])
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01
    )
    
    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        output_dir=OUTPUT_DIR,
        patience=PATIENCE
    )
    
    # Start training
    trainer.train(num_epochs=NUM_EPOCHS)
    
    print("\nTraining script complete!")


Using device: cuda and Cuda Version: 13.0

Loading preprocessed data from c:\Users\jacst\Downloads\School\Fall25\CSE575/Preprocessed/preprocessed_data.pkl...
    Total epochs: (1368, 480000)
    Total labels: (1368,)

Initialized SleepApneaDataset:
    Total epochs: 1368
    Valid sequences: 1342
    Context epochs: 14
    Output epochs: 10

Dataset split:
    Training samples: 1074
    Validation samples: 268

Initializing model...

Initializing MultiEpochSleepApneaDetector: 
    Context Epochs: 14
    Output Epochs: 10
    Number of Classes: 3
    Dropout: 0.3

Model initialized successfully.
AST Feature Dimension: 768

Initialized WeightedCrossEntropyLoss with weights: tensor([1.0000, 1.3000, 2.1000])

Initialized Trainer:
    Device: cuda
    Output Directory: c:\Users\jacst\Downloads\School\Fall25\CSE575\Model_Output
    Training Batches: 1074
    Val Batches: 268
    Patience: 10 epochs

Starting training for 1 epochs...

Epoch 1/1
----------------------------------------


Epoch 1 [Training]:   0%|          | 0/1074 [00:00<?, ?it/s]